# LLM-Generated Synthetic Data

## What This Is
Large language models can generate synthetic data via instruction following: describe the distribution you want, the model generates samples. This is particularly useful for:
- **Text classification training data**: "Generate 100 examples of customer complaints about billing"
- **Rare event generation**: "Write 50 examples of phishing emails that evade basic keyword filters"
- **Data augmentation with context**: LLMs can generate semantically coherent variations

## Quality Filtering
Raw LLM-generated data contains noise: off-topic samples, repetitions, degenerate outputs. A quality filter pipeline:
1. **Self-consistency**: Generate multiple versions; keep only those the model itself confidently classifies correctly
2. **Perplexity filtering**: Remove samples with very high or very low perplexity (incoherent or too templated)
3. **Diversity filtering**: Remove near-duplicate samples (SimHash, embedding deduplication)
4. **Label verification**: Use a held-out classifier to verify the generated sample has the intended label

## Risks
- **Hallucinated statistics**: LLM-generated numerical data may not match real distributions
- **Copyright/PII leakage**: LLMs trained on web data may reproduce memorized real records
- **Stereotype amplification**: LLMs amplify biases in their training data

In [ ]:
import numpy as np
import random
import hashlib
from typing import List, Dict, Tuple

np.random.seed(42)
random.seed(42)

# Simulate LLM-generated text data with a quality filter pipeline
# In practice: replace generate_llm_samples with actual API calls

SENTIMENT_TEMPLATES = {
    'positive': [
        'The product exceeded my expectations completely.',
        'Absolutely fantastic service and quality.',
        'I love this item, would highly recommend it.',
        'Outstanding value for the price paid.',
        'Best purchase I have made this year.',
        'The quality is remarkable and delivery was fast.',
        'Exceptional product that works exactly as described.',
    ],
    'negative': [
        'Complete waste of money, totally disappointed.',
        'Arrived broken and customer service was unhelpful.',
        'Nothing like the description, very misleading.',
        'Poor quality, fell apart after one week.',
        'Terrible experience from start to finish.',
        'Would not recommend to anyone, very bad.',
        'Stopped working immediately, refund denied.',
    ],
    'neutral': [
        'Product arrived on time, as described.',
        'Average quality for the price.',
        'Does what it says, nothing more.',
        'Shipping was fine, product is okay.',
        'Met basic expectations, nothing special.',
    ],
}

def generate_llm_samples(label: str, n: int, noise_rate: float = 0.15) -> List[str]:
    """Simulate LLM generation with noise (wrong label samples, duplicates)."""
    templates = SENTIMENT_TEMPLATES[label]
    all_templates = [t for templates_list in SENTIMENT_TEMPLATES.values() for t in templates_list]
    
    samples = []
    for i in range(n):
        if random.random() < noise_rate:
            # Inject noise: wrong label sample
            wrong_labels = [l for l in SENTIMENT_TEMPLATES if l != label]
            wrong_label = random.choice(wrong_labels)
            samples.append(random.choice(SENTIMENT_TEMPLATES[wrong_label]))
        else:
            base = random.choice(templates)
            # Simple variation: word substitution
            words = base.split()
            if len(words) > 4 and random.random() < 0.3:
                idx = random.randint(1, len(words)-2)
                words[idx] = words[idx] + 's'  # minor variation
            samples.append(' '.join(words))
    
    return samples

# Generate raw LLM data
raw_data = []
raw_labels = []
for label in ['positive', 'negative', 'neutral']:
    samples = generate_llm_samples(label, 50, noise_rate=0.15)
    raw_data.extend(samples)
    raw_labels.extend([label] * 50)

print(f'Raw LLM-generated samples: {len(raw_data)}')
print(f'Sample positive: "{raw_data[0]}"')
print(f'Sample negative: "{raw_data[50]}"')


In [ ]:
# Quality filter pipeline

class LLMDataQualityFilter:
    def __init__(self):
        self.stats = {}
    
    def dedup_simhash(self, texts: List[str]) -> List[bool]:
        """Remove near-duplicates using simhash-like fingerprinting."""
        seen_hashes = set()
        keep = []
        for text in texts:
            # Simple word-level fingerprint
            words = set(text.lower().split())
            fp = frozenset(list(words)[:8])  # first 8 unique words
            if fp not in seen_hashes:
                seen_hashes.add(fp)
                keep.append(True)
            else:
                keep.append(False)
        return keep
    
    def length_filter(self, texts: List[str], min_words: int = 5, max_words: int = 50) -> List[bool]:
        """Remove too-short or too-long samples."""
        return [min_words <= len(t.split()) <= max_words for t in texts]
    
    def label_consistency_filter(
        self, texts: List[str], labels: List[str],
        label_keywords: Dict[str, List[str]]
    ) -> List[bool]:
        """Check if text contains keywords consistent with its label."""
        keep = []
        for text, label in zip(texts, labels):
            text_lower = text.lower()
            expected_kw = label_keywords.get(label, [])
            has_expected = any(kw in text_lower for kw in expected_kw)
            has_opposite = False
            for other_label, kws in label_keywords.items():
                if other_label != label:
                    if any(kw in text_lower for kw in kws):
                        has_opposite = True
            keep.append(has_expected and not has_opposite)
        return keep
    
    def filter(self, texts: List[str], labels: List[str]) -> Tuple[List[str], List[str], Dict]:
        n = len(texts)
        
        # Length filter
        lf = self.length_filter(texts)
        
        # Dedup
        dd = self.dedup_simhash(texts)
        
        # Label keywords
        keywords = {
            'positive': ['great', 'excellent', 'fantastic', 'love', 'outstanding', 'best', 'exceptional'],
            'negative': ['terrible', 'broken', 'waste', 'disappointed', 'poor', 'bad', 'refused'],
            'neutral': ['average', 'okay', 'fine', 'basic', 'met'],
        }
        lc = self.label_consistency_filter(texts, labels, keywords)
        
        # Combine: keep if passes ALL filters
        keep_mask = [a and b for a, b in zip(lf, dd)]
        
        filtered_texts = [t for t, k in zip(texts, keep_mask) if k]
        filtered_labels = [l for l, k in zip(labels, keep_mask) if k]
        
        stats = {
            'original': n,
            'after_length': sum(lf),
            'after_dedup': sum(dd),
            'final': len(filtered_texts),
            'retention_rate': len(filtered_texts) / n,
        }
        return filtered_texts, filtered_labels, stats

qf = LLMDataQualityFilter()
clean_texts, clean_labels, stats = qf.filter(raw_data, raw_labels)

print('Quality Filter Results:')
for k, v in stats.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.3f}')
    else:
        print(f'  {k}: {v}')

print(f'\nLabel distribution (after filtering):')
for label in ['positive', 'negative', 'neutral']:
    count = clean_labels.count(label)
    print(f'  {label}: {count}')
